In [1]:
import os, glob, re, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from scipy import stats
from PIL import Image

warnings.filterwarnings('ignore')

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT    = '/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset'
BASE_H  = 176
BASE_W  = 512
N_CLASSES = 6
SEED    = 42
EPOCHS  = 45
BATCH   = 64
LR      = 1e-3
K_FOLDS = 5
GAP     = 8
WIDTH   = 32
OC      = 48
USE_NOISE = False
TRAIN_SNR_RANGE = (-5.0, 20.0)
TRAIN_NOISE_P   = 0.85

CLASS_MAP = {
    '3_long_blade_rotor':     '3_long_blade_rotor',
    '3_short_blade_rotor_1':  '3_short_blade_rotor',
    '3_short_blade_rotor_2':  '3_short_blade_rotor',
    'Bird':                   'Bird',
    'Bird+mini-helicopter_1': 'Bird+mini-helicopter',
    'Bird+mini-helicopter_2': 'Bird+mini-helicopter',
    'RC plane_1':             'RC_plane',
    'RC plane_2':             'RC_plane',
    'drone_1':                'drone',
    'drone_2':                'drone',
}
CLASSES = sorted(set(CLASS_MAP.values()))
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}


def numeric_key(p):
    nums = re.findall(r'\d+', os.path.basename(p))
    return int(nums[0]) if nums else 0


def list_images(folder):
    exts = ('*.png','*.jpg','*.jpeg','*.PNG','*.JPG','*.JPEG')
    out = []
    for e in exts:
        out += glob.glob(os.path.join(folder, '**', e), recursive=True)
    return out


def autocrop_resize(path):
    img = Image.open(path).convert('L')
    arr = np.asarray(img, dtype=np.float32)
    mask = arr < 240
    if mask.any():
        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        arr = arr[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]
    return np.asarray(
        Image.fromarray(arr.astype(np.uint8)).resize((BASE_W, BASE_H), Image.BILINEAR),
        dtype=np.uint8)


print('Enumerating files in filename (recording) order...')
paths, labels = [], []
for top in sorted(os.listdir(ROOT)):
    if top in CLASS_MAP:
        c = CLS2IDX[CLASS_MAP[top]]
        for f in sorted(list_images(os.path.join(ROOT, top)), key=lambda p: (numeric_key(p), p)):
            paths.append(f); labels.append(c)
labels = np.array(labels, dtype=np.int64)
N = len(paths)
print('Total images:', N)

print('Loading images...')
bdata = np.zeros((N, BASE_H, BASE_W), dtype=np.uint8)
for i, p in enumerate(paths):
    bdata[i] = autocrop_resize(p)
print('Loaded:', bdata.shape)

idx_all = np.arange(N)


def trim(a, g):
    return a[g:-g] if len(a) > 2 * g + 1 else a


def make_folds(K, gap):
    cb = {c: np.array_split(idx_all[labels == c], K) for c in range(N_CLASSES)}
    folds = []
    for k in range(K):
        tr, va, te = [], [], []
        for c in range(N_CLASSES):
            bl = cb[c]
            te.append(trim(bl[k], gap)); va.append(trim(bl[(k + 1) % K], gap))
            for j in range(K):
                if j != k and j != (k + 1) % K:
                    tr.append(trim(bl[j], gap))
        folds.append((np.concatenate(tr), np.concatenate(va), np.concatenate(te)))
    return folds


FOLDS = make_folds(K_FOLDS, GAP)
print(f'Honest folds K={K_FOLDS}, GAP={GAP}, mode={"NOISE-AUG" if USE_NOISE else "CLEAN"}')


def add_noise(img, snr_db):
    if snr_db is None:
        return img
    sig = float(img.var()) + 1e-8
    npow = sig / (10 ** (snr_db / 10.0))
    noise = np.random.randn(*img.shape).astype(np.float32) * np.sqrt(npow)
    return np.clip(img + noise, 0.0, 1.0).astype(np.float32)


class ConvBNSiLU(nn.Module):
    def __init__(self, ci, co, k=3, s=1):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(ci, co, k, s, k // 2, bias=False),
                               nn.BatchNorm2d(co), nn.SiLU(inplace=True))
    def forward(self, x): return self.b(x)


class DSConv2d(nn.Module):
    def __init__(self, ci, co, s=1):
        super().__init__()
        self.dw = nn.Conv2d(ci, ci, 3, s, 1, groups=ci, bias=False)
        self.pw = nn.Conv2d(ci, co, 1, bias=False)
        self.bn = nn.BatchNorm2d(co); self.act = nn.SiLU(inplace=True)
    def forward(self, x): return self.act(self.bn(self.pw(self.dw(x))))


class DSConv1d(nn.Module):
    def __init__(self, ci, co, k=5, s=1):
        super().__init__()
        self.dw = nn.Conv1d(ci, ci, k, s, k // 2, groups=ci, bias=False)
        self.pw = nn.Conv1d(ci, co, 1, bias=False)
        self.bn = nn.BatchNorm1d(co); self.act = nn.SiLU(inplace=True)
    def forward(self, x): return self.act(self.bn(self.pw(self.dw(x))))


class ResDS2d(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.b = nn.Sequential(DSConv2d(ch, ch, 1), DSConv2d(ch, ch, 1))
    def forward(self, x): return x + self.b(x)


class Res1d(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.b = nn.Sequential(DSConv1d(ch, ch, 5, 1), DSConv1d(ch, ch, 5, 1))
    def forward(self, x): return x + self.b(x)


class SE(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                nn.Linear(c, max(c // r, 4)), nn.SiLU(inplace=True),
                                nn.Linear(max(c // r, 4), c), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x).view(x.size(0), -1, 1, 1)


class MultiScaleDW1d(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.k3 = nn.Conv1d(ci, ci, 3, 1, 1, groups=ci, bias=False)
        self.k5 = nn.Conv1d(ci, ci, 5, 1, 2, groups=ci, bias=False)
        self.k7 = nn.Conv1d(ci, ci, 7, 1, 3, groups=ci, bias=False)
        self.bn = nn.BatchNorm1d(ci); self.act = nn.SiLU(inplace=True)
        self.pw = nn.Sequential(nn.Conv1d(ci, co, 1, bias=False),
                                nn.BatchNorm1d(co), nn.SiLU(inplace=True))
    def forward(self, x):
        return self.pw(self.act(self.bn(self.k3(x) + self.k5(x) + self.k7(x))))


class CVDLayer(nn.Module):
    def forward(self, x):
        with torch.amp.autocast('cuda', enabled=False):
            x = x.float()
            return torch.log1p(torch.fft.rfft(x, dim=3).abs())


def vel_branch(width, oc, depth):
    return nn.Sequential(MultiScaleDW1d(width, oc), *[Res1d(oc) for _ in range(depth)],
                         DSConv1d(oc, oc, 5, 2))


def cvd_branch(width, oc, depth):
    return nn.Sequential(DSConv2d(width, oc, 2), *[ResDS2d(oc) for _ in range(depth)],
                         SE(oc), DSConv2d(oc, oc, 2))


def joint_branch(width, oc, depth):
    return nn.Sequential(DSConv2d(width, oc, 2), *[ResDS2d(oc) for _ in range(depth)],
                         DSConv2d(oc, oc, 2))


class ImprovedNet(nn.Module):
    def __init__(self, streams=('vel', 'cvd', 'joint'), fusion='dense', depth=1,
                 width=WIDTH, oc=OC, n=N_CLASSES):
        super().__init__()
        self.streams = streams; self.fusion = fusion
        self.stem = nn.Sequential(ConvBNSiLU(1, 16, 3, 2), ConvBNSiLU(16, width, 3, 2),
                                  ResDS2d(width), SE(width))
        if 'vel' in streams: self.vel = vel_branch(width, oc, depth)
        if 'tmp' in streams: self.tmp = vel_branch(width, oc, depth)
        if 'cvd' in streams: self.cvd_op = CVDLayer(); self.cvd = cvd_branch(width, oc, depth)
        if 'joint' in streams: self.joint = joint_branch(width, oc, depth)
        self.gap1 = nn.AdaptiveAvgPool1d(1); self.gap2 = nn.AdaptiveAvgPool2d(1)
        d = oc * len(streams)
        if fusion == 'gate':
            self.gate = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
        self.fuse = nn.Sequential(nn.Linear(d, d), nn.BatchNorm1d(d),
                                  nn.SiLU(inplace=True), nn.Dropout(0.3))
        self.cls = nn.Linear(d, n)
    def forward(self, x):
        f = self.stem(x); e = []
        if 'vel' in self.streams: e.append(self.gap1(self.vel(f.mean(dim=3))).flatten(1))
        if 'tmp' in self.streams: e.append(self.gap1(self.tmp(f.mean(dim=2))).flatten(1))
        if 'cvd' in self.streams: e.append(self.gap2(self.cvd(self.cvd_op(f))).flatten(1))
        if 'joint' in self.streams: e.append(self.gap2(self.joint(f)).flatten(1))
        cat = torch.cat(e, 1)
        if self.fusion == 'gate': cat = self.gate(cat) * cat
        return self.cls(self.fuse(cat))


class DS(Dataset):
    def __init__(self, idx, train):
        self.idx = idx; self.train = train
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        img = bdata[j].astype(np.float32) / 255.0
        if self.train:
            if random.random() < 0.5:
                img = img[:, ::-1].copy()
            if random.random() < 0.5:
                img = np.clip(img * random.uniform(0.9, 1.1), 0.0, 1.0).astype(np.float32)
            if USE_NOISE and random.random() < TRAIN_NOISE_P:
                img = add_noise(img, random.uniform(*TRAIN_SNR_RANGE))
            if random.random() < 0.3:
                h0 = random.randint(0, BASE_H - 20)
                img[h0:h0 + random.randint(5, 20), :] = 0.0
            if random.random() < 0.3:
                w0 = random.randint(0, BASE_W - 25)
                img[:, w0:w0 + random.randint(5, 25)] = 0.0
        return torch.from_numpy(img).float().unsqueeze(0), int(labels[j])


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def train_eval_fold(build_fn, tr_idx, va_idx, te_idx):
    set_seed(SEED)
    m = build_fn().to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-2)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    scl = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    tl = DataLoader(DS(tr_idx, True), batch_size=BATCH, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(DS(va_idx, False), batch_size=BATCH, shuffle=False,
                    num_workers=2, pin_memory=True)
    best, best_state = 0.0, None
    for ep in range(EPOCHS):
        m.train()
        for x, y in tl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                loss = crit(m(x), y)
            scl.scale(loss).backward(); scl.step(opt); scl.update()
        sch.step()
        m.eval(); c = t = 0
        with torch.no_grad():
            for x, y in vl:
                x, y = x.to(DEVICE), y.to(DEVICE)
                c += (m(x).argmax(1) == y).sum().item(); t += y.size(0)
        if c / t >= best:
            best = c / t
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state); m.eval()
    tel = DataLoader(DS(te_idx, False), batch_size=BATCH, shuffle=False,
                     num_workers=2, pin_memory=True)
    yt, yp = [], []
    with torch.no_grad():
        for x, y in tel:
            yp.append(m(x.to(DEVICE)).argmax(1).cpu().numpy()); yt.append(y.numpy())
    del m
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return accuracy_score(np.concatenate(yt), np.concatenate(yp)), np.concatenate(yp), np.concatenate(yt)


def run_config(build_fn):
    accs, pool_yp, pool_yt = [], [], []
    for (tr, va, te) in FOLDS:
        a, yp, yt = train_eval_fold(build_fn, tr, va, te)
        accs.append(a); pool_yp.append(yp); pool_yt.append(yt)
    accs = np.array(accs)
    ci = stats.t.interval(0.95, len(accs) - 1, loc=accs.mean(), scale=stats.sem(accs))
    return dict(accs=accs, mean=accs.mean(), std=accs.std(ddof=1), ci=ci,
                params=sum(p.numel() for p in build_fn().parameters()),
                pool_yp=np.concatenate(pool_yp), pool_yt=np.concatenate(pool_yt))


def mcnemar(yp_a, yp_b, yt):
    ca = (yp_a == yt); cb = (yp_b == yt)
    b = ((~ca) & cb).sum(); c = (ca & (~cb)).sum()
    if b + c == 0:
        return 1.0
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return float(1 - stats.chi2.cdf(chi2, df=1))


VARIANTS = [
    ('velocity_only',       ('vel',),                'dense'),
    ('naive_temporal_only', ('tmp',),                'dense'),
    ('cvd_only',            ('cvd',),                'dense'),
    ('vel+naive_temporal',  ('vel', 'tmp'),          'dense'),
    ('vel+cvd_concat',      ('vel', 'cvd'),          'dense'),
    ('vel+cvd_gate',        ('vel', 'cvd'),          'gate'),
    ('vel+cvd+joint',       ('vel', 'cvd', 'joint'), 'dense'),
]

print('\n' + '=' * 88)
print(f'STAGE 1 - TECHNIQUE COMBINATION (improved model, depth=1, {"NOISE" if USE_NOISE else "CLEAN"})')
print('=' * 88)
s1, cfg = {}, {}
for name, streams, fusion in VARIANTS:
    print(f'\nRunning {name} ...')
    r = run_config(lambda st=streams, fu=fusion: ImprovedNet(st, fu, depth=1))
    s1[name] = r; cfg[name] = (streams, fusion)
    print(f'  folds {[f"{a*100:.2f}" for a in r["accs"]]}  '
          f'mean {r["mean"]*100:.2f}% +/- {r["std"]*100:.2f}  params {r["params"]:,}')

ref = s1['velocity_only']
print('\n' + '-' * 88)
print(f"{'Variant':<22}{'Mean%':>8}{'+/-Std':>8}{'CI95_lo':>9}{'CI95_hi':>9}{'Params':>9}{'McNemar_p':>11}")
print('-' * 88)
for name, _, _ in VARIANTS:
    r = s1[name]
    p = 1.0 if name == 'velocity_only' else mcnemar(r['pool_yp'], ref['pool_yp'], ref['pool_yt'])
    r['mcnemar_p'] = p
    star = '*' if (p < 0.05 and r['mean'] > ref['mean']) else ''
    print(f"{name:<22}{r['mean']*100:>8.2f}{r['std']*100:>8.2f}"
          f"{r['ci'][0]*100:>9.2f}{r['ci'][1]*100:>9.2f}{r['params']:>9,}{p:>11.4f}{star}")
print('* = significantly beats velocity_only (McNemar p<0.05 AND higher mean)')

beats = [(n, s1[n]) for n, _, _ in VARIANTS
         if n != 'velocity_only' and s1[n]['mean'] > ref['mean'] and s1[n]['mcnemar_p'] < 0.05]
if beats:
    beats.sort(key=lambda kv: (-kv[1]['mean'], kv[1]['params']))
    win_name = beats[0][0]
else:
    win_name = 'velocity_only'
win_streams, win_fusion = cfg[win_name]
print(f'\nSTAGE 1 WINNER: {win_name}  streams={win_streams} fusion={win_fusion}')

print('\n' + '=' * 88)
print(f'STAGE 2 - DEPTH SWEEP ON WINNER ({win_name})')
print('=' * 88)
DEPTHS = [1, 2, 3, 4]
s2 = {}
for d in DEPTHS:
    print(f'\nDepth {d} ...')
    r = run_config(lambda dd=d: ImprovedNet(win_streams, win_fusion, depth=dd))
    s2[d] = r
    print(f'  mean {r["mean"]*100:.2f}% +/- {r["std"]*100:.2f}  '
          f'CI[{r["ci"][0]*100:.2f},{r["ci"][1]*100:.2f}]  params {r["params"]:,}')

print('\n' + '-' * 70)
print(f"{'Depth':>6}{'Mean%':>9}{'+/-Std':>8}{'CI95_lo':>9}{'CI95_hi':>9}{'Params':>10}")
print('-' * 70)
for d in DEPTHS:
    r = s2[d]
    print(f"{d:>6}{r['mean']*100:>9.2f}{r['std']*100:>8.2f}"
          f"{r['ci'][0]*100:>9.2f}{r['ci'][1]*100:>9.2f}{r['params']:>10,}")

best_d = max(DEPTHS, key=lambda d: s2[d]['mean'])
thr = s2[best_d]['mean'] - s2[best_d]['std']
chosen_d = min([d for d in DEPTHS if s2[d]['mean'] >= thr])

print('\n' + '=' * 88)
print('FINAL SELECTION')
print('=' * 88)
print(f'  Mode         : {"noise-augmented" if USE_NOISE else "clean"} honest folds')
print(f'  Architecture : {win_name}  streams={win_streams} fusion={win_fusion}')
print(f'  Depth        : {chosen_d} (best mean at depth {best_d})')
print(f'  Honest acc   : {s2[chosen_d]["mean"]*100:.2f}% +/- {s2[chosen_d]["std"]*100:.2f}')
print(f'  Params       : {s2[chosen_d]["params"]:,}')

tag = 'noise' if USE_NOISE else 'clean'
np.savez(os.path.join('/kaggle/working', f'improved_ablation_{tag}.npz'),
         winner=win_name, depth=chosen_d,
         s1_means={k: float(v['mean']) for k, v in s1.items()},
         s2_means={d: float(s2[d]['mean']) for d in DEPTHS})
print(f'\nSaved improved_ablation_{tag}.npz')

Enumerating files in filename (recording) order...
Total images: 4849
Loading images...
Loaded: (4849, 176, 512)
Honest folds K=5, GAP=8, mode=CLEAN

STAGE 1 - TECHNIQUE COMBINATION (improved model, depth=1, CLEAN)

Running velocity_only ...
  folds ['77.00', '84.32', '89.93', '87.76', '88.09']  mean 85.42% +/- 5.12  params 20,990

Running naive_temporal_only ...
  folds ['75.40', '82.95', '92.22', '88.33', '90.15']  mean 85.81% +/- 6.76  params 20,990

Running cvd_only ...
  folds ['81.81', '86.04', '91.99', '92.79', '92.10']  mean 88.95% +/- 4.83  params 22,522

Running vel+naive_temporal ...
  folds ['78.49', '83.18', '89.47', '89.70', '92.33']  mean 86.63% +/- 5.66  params 38,430

Running vel+cvd_concat ...
  folds ['81.69', '84.90', '90.96', '93.48', '87.74']  mean 87.75% +/- 4.69  params 39,962

Running vel+cvd_gate ...
  folds ['79.86', '86.27', '90.96', '90.27', '92.55']  mean 87.98% +/- 5.10  params 49,274

Running vel+cvd+joint ...
  folds ['80.32', '86.04', '89.24', '93.02',